In [1]:
import numpy as np
import pandas as pd
from scipy.interpolate import Rbf
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

import time
import os

# Load table
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_with_country_names.csv')
lsms_spatial = lsms_spatial[['x', 'y', 'country', 'farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita', 'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market']]
lsms_spatial = lsms_spatial.dropna()
print(lsms_spatial.columns)

# Customized functions
def test_tps(d):
    Zvars = ["cropland", "cattle", "pop", "cropland_per_capita", "sand", "slope", "temperature", "rainfall", "market", "maizeyield"]
    Z = d[Zvars].values
    tps_model = Rbf(d['x'], d['y'], d['farm_area_ha'], function='thin_plate', smooth=0)
    prediction = tps_model(d['x'], d['y'])
    # rsq = r2_score(d['farm_area_ha'], prediction)  # this is not the R square, a score just by the name
    correlation_coef = np.corrcoef(d['farm_area_ha'], prediction)[0, 1]
    rsq = correlation_coef ** 2
    return {'prediction': prediction, 'rsq': rsq}

def test_rf(d_train, d_test):
    rf_model = RandomForestRegressor(n_estimators=100, random_state=2024)
    X_train = d_train.drop(columns=['x', 'y', 'farm_area_ha'])
    y_train = d_train['farm_area_ha']
    rf_model.fit(X_train, y_train)
    X_test = d_test.drop(columns=['x', 'y', 'farm_area_ha'])
    prediction = rf_model.predict(X_test)
    correlation_coef = np.corrcoef(d_test['farm_area_ha'], prediction)[0, 1]
    rsq = correlation_coef ** 2
    return {'prediction': prediction, 'rsq': rsq}

def leave_one_country_models(the_country, the_code, model, means, test, sample_size=None):
    assert model in ["TPS", "RF"]
    input_path = "../data/processed"
    output_path = "../output/leave_one"
    os.makedirs(output_path, exist_ok=True)
    
    print(f"--------------- Model evaluation in {the_country} -------------")
    
    fname_rsq = os.path.join(output_path, f"loc_{the_code}_{model}_{'all' if not means else 'means'}_{'train' if not test else 'test'}_rsq.csv")
    fname_pred = os.path.join(output_path, f"loc_{the_code}_{model}_{'all' if not means else 'means'}_{'train' if not test else 'test'}_pred.csv")
    
    np.random.seed(2024)  # for reproducibility
    
    training_set = lsms_spatial[lsms_spatial['country'] != the_country].drop(columns=['country']).dropna()
    test_set = lsms_spatial[lsms_spatial['country'] == the_country].drop(columns=['country']).dropna()
    
    if means:
        training_set_mean = (
            training_set
            .groupby(['x', 'y'])
            .agg(
                n_obs=('x', 'size'),
                **{col: (col, lambda x: x.mean(skipna=True)) for col in training_set.select_dtypes(include=['float', 'int']).columns}
            )
            .reset_index(drop=True)
        )
        training_set_mean = training_set_mean[training_set_mean['n_obs'] > 9].drop(columns=['n_obs'])
        
        test_set_mean = (
            test_set
            .groupby(['x', 'y'])
            .agg(
                n_obs=('x', 'size'),
                **{col: (col, lambda x: x.mean(skipna=True)) for col in test_set.select_dtypes(include=['float', 'int']).columns}
            )
            .reset_index(drop=True)
        )
        test_set_mean = test_set_mean[test_set_mean['n_obs'] > 9].drop(columns=['n_obs'])
        
        if model == "TPS":
            out = test_tps(test_set_mean)
        else:
            if test:
                out = test_rf(test_set_mean, test_set_mean)
            else:
                out = test_rf(training_set_mean, test_set_mean)
                
    else:
        if sample_size:
            training_set = training_set.sample(n=min(len(training_set), sample_size))
            test_set = test_set.sample(n=min(len(test_set), 2 * sample_size))
        
        if model == "TPS":
            out = test_tps(test_set)
        else:
            if test:
                out = test_rf(test_set, test_set)
            else:
                out = test_rf(training_set, test_set)
    
    # Save the r-squared results to a CSV file
    rsq_results = pd.DataFrame({
        'country': [the_country],
        'code': [the_code],
        'model': [model],
        'means': [means],
        'test': [test],
        'rsq': [out['rsq']]
    })
    rsq_results.to_csv(fname_rsq, index=False)
    
    # Save the predictions to a CSV file
    pred_results = pd.DataFrame({
        'prediction': out['prediction']
    })
    
    pred_results.to_csv(fname_pred, index=False)
    
    return fname_rsq, fname_pred

def summarize():
    rf_files_pred = [os.path.join("../output/leave_one", f) for f in os.listdir("../output/leave_one") if "RF" in f and f.endswith("_pred.csv")]
    tps_files_pred = [os.path.join("../output/leave_one", f) for f in os.listdir("../output/leave_one") if "TPS" in f and f.endswith("_pred.csv")]
    
    country_codes = ["BEN", "BFA", "CIV", "ETH", "GHA", "GNB", "MWI", "MLI", "NER", "NGA", "RWA", "SEN", "TZA", "TGO", "UGA", "ZMB"]
    
    cor_results = []

    for code in country_codes:
        # Filter files for the current country
        rf_test_files = [f for f in rf_files_pred if f"_{code}_" in f and "_RF_means_test_pred" in f]
        rf_train_files = [f for f in rf_files_pred if f"_{code}_" in f and "_RF_means_train_pred" in f]
        tps_test_files = [f for f in tps_files_pred if f"_{code}_" in f and "_TPS_means_test_pred" in f]

        if not rf_test_files or not rf_train_files or not tps_test_files:
            print(f"Missing files for country code: {code}")
            continue

        # Read the prediction data
        rf_test_df = pd.read_csv(rf_test_files[0])
        rf_train_df = pd.read_csv(rf_train_files[0])
        tps_test_df = pd.read_csv(tps_test_files[0])

        # Check if 'prediction' column exists
        if 'prediction' not in rf_test_df.columns or 'prediction' not in rf_train_df.columns or 'prediction' not in tps_test_df.columns:
            print(f"Missing 'prediction' column in files for country code: {code}")
            continue

        # Compute correlation coefficients
        cor_coef_1 = np.corrcoef(tps_test_df['prediction'], rf_train_df['prediction'])[0, 1]
        cor_coef_2 = np.corrcoef(rf_test_df['prediction'], rf_train_df['prediction'])[0, 1]
        print(f"cor_TPS for {code} is: {cor_coef_1}")
        print(f"cor_RFtrain for {code} is: {cor_coef_2}")


        cor_results.append({
            'country': code,
            'cor_coef_TPS_vs_RFother_countries': cor_coef_1,
            'cor_coef_RF_vs_RFother_countries': cor_coef_2
        })

    cor_results_df = pd.DataFrame(cor_results)
    cor_results_df.to_csv("../output/leave_one/leave_one_cor.csv", index=False)

countries = ["Benin", "Burkina", "Cote_d_Ivoire", "Ethiopia", "Ghana", "Guinea_Bissau", "Malawi", "Mali", "Niger", "Nigeria", "Rwanda", "Senegal", "Tanzania", "Togo", "Uganda", "Zambia"]
country_codes = ["BEN", "BFA", "CIV", "ETH", "GHA", "GNB", "MWI", "MLI", "NER", "NGA", "RWA", "SEN", "TZA", "TGO", "UGA", "ZMB"]

trts = pd.DataFrame([(i, model, means, test) for i in range(16) for model in ["RF", "TPS"] for means in [True] for test in [True, False]], columns=['country', 'model', 'means', 'test'])
trts = trts[~((trts['model'] == "TPS") & (~trts['test']))]

# Sequential execution for all countries
deb = time.time()
for i in range(len(trts)):
    leave_one_country_models(countries[trts.iloc[i]['country']], country_codes[trts.iloc[i]['country']], trts.iloc[i]['model'], trts.iloc[i]['means'], trts.iloc[i]['test'])
    print(f"Processed country index: {i}")

fin = time.time()
print(f'Time to produce rsquares: {fin - deb} seconds')

# Summarize results
start_time = time.time()
summarize() 
end_time = time.time()
print(f'Time to produce cor coef: {end_time - start_time} seconds')

Index(['x', 'y', 'country', 'farm_area_ha', 'cropland', 'cattle', 'pop',
       'cropland_per_capita', 'sand', 'slope', 'temperature', 'rainfall',
       'maizeyield', 'market'],
      dtype='object')
--------------- Model evaluation in Benin -------------
Processed country index: 0
--------------- Model evaluation in Benin -------------
Processed country index: 1
--------------- Model evaluation in Benin -------------
Processed country index: 2
--------------- Model evaluation in Burkina -------------
Processed country index: 3
--------------- Model evaluation in Burkina -------------
Processed country index: 4
--------------- Model evaluation in Burkina -------------
Processed country index: 5
--------------- Model evaluation in Cote_d_Ivoire -------------
Processed country index: 6
--------------- Model evaluation in Cote_d_Ivoire -------------
Processed country index: 7
--------------- Model evaluation in Cote_d_Ivoire -------------
Processed country index: 8
--------------- Model 

C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\interpolate\_rbf.py:268: LinAlgWarning: Ill-conditioned matrix (rcond=1.42448e-18): result may not be accurate.
  self.nodes = linalg.solve(self.A, self.di)


Processed country index: 11
--------------- Model evaluation in Ghana -------------
Processed country index: 12
--------------- Model evaluation in Ghana -------------
Processed country index: 13
--------------- Model evaluation in Ghana -------------
Processed country index: 14
--------------- Model evaluation in Guinea_Bissau -------------
Processed country index: 15
--------------- Model evaluation in Guinea_Bissau -------------
Processed country index: 16
--------------- Model evaluation in Guinea_Bissau -------------
Processed country index: 17
--------------- Model evaluation in Malawi -------------
Processed country index: 18
--------------- Model evaluation in Malawi -------------
Processed country index: 19
--------------- Model evaluation in Malawi -------------


C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\interpolate\_rbf.py:268: LinAlgWarning: Ill-conditioned matrix (rcond=1.24757e-19): result may not be accurate.
  self.nodes = linalg.solve(self.A, self.di)


Processed country index: 20
--------------- Model evaluation in Mali -------------
Processed country index: 21
--------------- Model evaluation in Mali -------------
Processed country index: 22
--------------- Model evaluation in Mali -------------
Processed country index: 23
--------------- Model evaluation in Niger -------------
Processed country index: 24
--------------- Model evaluation in Niger -------------
Processed country index: 25
--------------- Model evaluation in Niger -------------
Processed country index: 26
--------------- Model evaluation in Nigeria -------------
Processed country index: 27
--------------- Model evaluation in Nigeria -------------
Processed country index: 28
--------------- Model evaluation in Nigeria -------------
Processed country index: 29
--------------- Model evaluation in Rwanda -------------
Processed country index: 30
--------------- Model evaluation in Rwanda -------------
Processed country index: 31
--------------- Model evaluation in Rwanda 

C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\interpolate\_rbf.py:268: LinAlgWarning: Ill-conditioned matrix (rcond=4.52771e-19): result may not be accurate.
  self.nodes = linalg.solve(self.A, self.di)


Processed country index: 38
--------------- Model evaluation in Togo -------------
Processed country index: 39
--------------- Model evaluation in Togo -------------
Processed country index: 40
--------------- Model evaluation in Togo -------------
Processed country index: 41
--------------- Model evaluation in Uganda -------------
Processed country index: 42
--------------- Model evaluation in Uganda -------------
Processed country index: 43
--------------- Model evaluation in Uganda -------------


C:\Users\DHOUGNI\AppData\Local\Programs\Python\Python313\Lib\site-packages\scipy\interpolate\_rbf.py:268: LinAlgWarning: Ill-conditioned matrix (rcond=3.7348e-18): result may not be accurate.
  self.nodes = linalg.solve(self.A, self.di)


Processed country index: 44
--------------- Model evaluation in Zambia -------------
Processed country index: 45
--------------- Model evaluation in Zambia -------------
Processed country index: 46
--------------- Model evaluation in Zambia -------------
Processed country index: 47
Time to produce rsquares: 1074.2389044761658 seconds
cor_TPS for BEN is: 0.7344502451786801
cor_RFtrain for BEN is: 0.7895196360048851
cor_TPS for BFA is: 0.316607105037662
cor_RFtrain for BFA is: 0.37623174520322156
cor_TPS for CIV is: 0.3922951630333129
cor_RFtrain for CIV is: 0.41090147244485237
cor_TPS for ETH is: 0.0941506415809269
cor_RFtrain for ETH is: 0.07154372371969035
cor_TPS for GHA is: 0.2047180993635659
cor_RFtrain for GHA is: 0.30923024802892557
cor_TPS for GNB is: -0.1845249096882716
cor_RFtrain for GNB is: -0.22816218588020942
cor_TPS for MWI is: 0.06533511332260217
cor_RFtrain for MWI is: 0.11996768522158505
cor_TPS for MLI is: 0.3361142457207776
cor_RFtrain for MLI is: 0.38187365541435414